In [1]:
# Import necessary libraries
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import time
import numpy as np

In [2]:
# Load breast cancer dataset
data = load_breast_cancer()
X = data.data
y = data.target


# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
# Definició de la graella de paràmetres (param_grid)
# L'objectiu és provar diferents combinacions per trobar la que millor prediu
param_grid = {
    # 'max_depth': Controla la profunditat de l'arbre.
    # Si és massa profund, l'arbre s'aprèn les dades de memòria (overfitting).
    # 'None' vol dir que l'arbre creixerà fins que totes les fulles siguin pures.
    'max_depth': [3, 5, 7, 9, None],

    # 'min_samples_split': El nombre mínim de mostres que ha de tenir un node
    # per poder dividir-se en dos de nous. Ajuda a evitar divisions basades en pocs casos.
    'min_samples_split': [2, 5, 10],

    # 'min_samples_leaf': El nombre mínim de mostres que han de quedar en una "fulla" (el final).
    # Valors més alts fan que el model sigui més simple i menys sensible al soroll.
    'min_samples_leaf': [1, 2, 4],

    # 'criterion': La funció per mesurar la qualitat d'una divisió.
    # 'gini' és més ràpid computacionalment; 'entropy' sol donar arbres una mica més equilibrats.
    'criterion': ['gini', 'entropy']
}

# Graella estesa (param_grid_ext)
# Aquesta versió és molt més granular (té més passos entremig)
param_grid_ext = {
    'max_depth': [3, 4, 5, 6, 7, 8, 9, None],
    'min_samples_split': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'min_samples_leaf': [1, 2, 3, 4, 5],
    'criterion': ['gini', 'entropy'],
    # Controla la "poda" de l'arbre. Valors petits com 0.01 poden fer meravelles.
    'ccp_alpha': [0.0, 0.001, 0.01, 0.02],

    # Quantes variables mira a cada divisió.
    # 'sqrt' sol anar molt bé per evitar que l'arbre sigui massa rígid.
    'max_features': [None, 'sqrt', 'log2']
}

## Grid Search

In [5]:
# 1. Creem l'instància del model base (Arbre de Decisió)
# random_state=42: Fixem la "llavor" perquè els resultats siguin sempre els mateixos cada cop que executis el codi.
dt_classifier = DecisionTreeClassifier(random_state=42)

# 2. Configurem la cerca exhaustiva (GridSearchCV)
grid_search = GridSearchCV(
    estimator=dt_classifier,    # El model que volem optimitzar.
    param_grid=param_grid_ext,  # El diccionari amb tots els valors que volem provar.

    # scoring: El criteri per decidir quin model és "millor".
    # 'accuracy' és el % d'encerts total, però podríem usar 'recall' si prioritzéssim detectar tots els càncers.
    scoring='accuracy',

    # cv: Estratègia de validació creuada.
    # StratifiedKFold assegura que cada "tros" (fold) tingui la mateixa proporció de tumors que el dataset original.
    cv=StratifiedKFold(n_splits=5),

    # n_jobs=-1: Fa servir tots els processadors de l'ordinador en paral·lel per anar més ràpid.
    n_jobs=-1,

    # verbose=1: Ens mostra missatges per pantalla mentre treballa (molt útil per saber quant falta).
    verbose=1
)

# 3. Iniciem l'entrenament de totes les combinacions
# Aquí és on GridSearchCV agafa el param_grid i entrena els 720 models possibles.
# GridSearch internament fa Cross-Validation per cada combinació per assegurar-se que els resultats són robustos.
# Hold out: No utilitza X_test en aquest pas, només X_train. X_test es reserva per avaluació final.
# Normalment s'aplica maxim 5-fold CV per cada combinació. Això vol dir que per cada combinació,
# el model s'entrena 5 vegades (una per cada "fold").
# Per tant, si tenim 720 combinacions i fem 5-fold CV, el total d'entrenaments serà 720 * 5 = 3600.
# Això pot trigar molt de temps depenent de la mida del dataset i la complexitat del model.
grid_search.fit(X_train, y_train)

# Si volem fer prunning per obtenir el millor arbre pero optim.
# Haurem d'afegir 'ccp_alpha' al param_grid_ext i deixar-lo en 0.01 o similar.

# 4. Extraiem els resultats guanyadors
# best_estimator_: És el model ja entrenat amb la millor configuració trobada.
best_dt_classifier = grid_search.best_estimator_
# best_params_: Ens diu quins valors concrets han funcionat millor (ex: max_depth=5).
best_params = grid_search.best_params_

# 5. Avaluació final amb dades que el model NO ha vist mai (X_test)
y_pred = best_dt_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on the test set: {accuracy:.4f}")
print("Best Parameters:", best_params)

Accuracy on the test set: 0.9211
Best Parameters: {'ccp_alpha': 0.01, 'criterion': 'gini', 'max_depth': 9, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 10}


## Random Search

In [6]:
# 1. Definició del classificador base
dt_classifier = DecisionTreeClassifier(random_state=42)

# 2. Configuració de la cerca aleatòria (RandomizedSearchCV)
random_search = RandomizedSearchCV(
    estimator=dt_classifier,
    # param_distributions: A diferència de GridSearch, aquí "distribuïm" les opcions.
    # No les provarà TOTES, sinó que en triarà algunes a l'atzar.
    param_distributions=param_grid_ext,

    # n_iter=50: Aquest és el paràmetre clau. Indica que només provarà 50 combinacions
    # aleatòries de les 720 possibles. Estalvia molt temps de computació.
    n_iter=50,

    # scoring: Mantenim 'accuracy' per mesurar l'èxit global.
    scoring='accuracy',

    # cv: Seguim usant StratifiedKFold perquè és el més segur per mantenir les proporcions de les classes.
    cv=StratifiedKFold(n_splits=5),

    # n_jobs=-1: Perquè el nostre processador treballi al màxim de la seva capacitat.
    n_jobs=-1,

    # verbose=1: Perquè ens vagi informant del progrés (quants "fits" porta fets).
    verbose=1,

    # random_state=42: Molt important aquí! Com que la tria és aleatòria, fixar aquest número
    # fa que, si tornes a executar el codi, triï exactament les mateixes 50 combinacions.
    random_state=42
)

# 3. Execució de la cerca
# El model tria 50 punts a l'atzar de la graella, els entrena i els compara.
random_search.fit(X_train, y_train)

# 4. Obtenció del millor model
best_dt_classifier = random_search.best_estimator_
best_params = random_search.best_params_

# 5. Predicció i càlcul de la precisió final
y_pred = best_dt_classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy on the test set: {accuracy:.4f}")
print("Best Parameters:", best_params)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Accuracy on the test set: 0.9035
Best Parameters: {'min_samples_split': 3, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 4, 'criterion': 'gini', 'ccp_alpha': 0.01}


## Bayes Search

In [8]:
from skopt import BayesSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# 1. Creem el classificador base
dt_classifier = DecisionTreeClassifier(random_state=42)

# 2. Configurem l'optimització Bayesiana (BayesSearchCV)
bayes_search = BayesSearchCV(
    estimator=dt_classifier,

    # search_spaces: A diferència del GridSearch (que és fix) o el Random (que és atzarós),
    # aquí definim l'espai que l'algoritme "explorarà" intel·ligentment.
    search_spaces=param_grid_ext,

    # n_iter=50: Farem 50 iteracions. L'algoritme usarà els resultats de la iteració 1
    # per decidir quins paràmetres provar a la iteració 2, i així successivament.
    n_iter=50,

    # scoring: Mantenim 'accuracy' per poder comparar amb els mètodes anteriors.
    scoring='accuracy',

    # cv: StratifiedKFold és essencial per garantir que cada "fold" sigui representatiu.
    cv=StratifiedKFold(n_splits=5),

    # n_jobs=-1: Paral·lelitza el procés (tot i que l'optimització Bayesiana és més
    # seqüencial per naturalesa, el càlcul de la Cross-Validation sí es pot accelerar).
    n_jobs=-1,

    # verbose=1: Ens anirà mostrant el progrés de la cerca.
    verbose=1,

    # random_state=42: Garanteix que l'exploració inicial i el soroll aleatori siguin replicables.
    random_state=42
)

# 3. Iniciem la cerca intel·ligent
# L'algoritme construeix un "model del model" (normalment un procés gaussià)
# per predir quins paràmetres donaran millors resultats.
bayes_search.fit(X_train, y_train)

# 4. Extraiem el millor model trobat
best_dt_classifier = bayes_search.best_estimator_
best_params = bayes_search.best_params_

# 5. Avaluem amb el test set
y_pred = best_dt_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on the test set: {accuracy:.4f}")
print("Best Parameters:", best_params)

Accuracy on the test set: 0.9474
Best Parameters: OrderedDict({'ccp_alpha': 0.02, 'criterion': 'entropy', 'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 5, 'min_samples_split': 2})
